In [0]:
import json
import math

# ============================================================
# Source Onboarding Agent tools.
# The agent proposes; these tools check, test, ask the human and only then act.
# All network/storage work is delegated to _utils_bronze_framework (including the API governance:
# budgets per quota group, usage log and blocks).
# ============================================================


def _no_data(sample):
    return sample is None or sample.strip() in ("", "[]", "{}", "null")


def _dry_run_window(spec):
    """Smallest window (in units) that still produces at least one chunk (sources with a delay need more)."""
    load = spec["load"]
    if load["strategy"] == "full":
        return None
    offsets = [load.get("end_offset", 0)] + [ep.get("end_offset", load.get("end_offset", 0)) for ep in spec["endpoints"]]
    return 1 + max(0, -min(offsets))


def quota_group_conflicts(spec):
    """Sources of the same quota group (same provider quota) must declare the same limits."""
    if not isinstance(spec.get("limits"), dict):
        return []
    new = source_limits(spec)
    errors = []
    for name in list_source_specs():
        other = load_source_spec_file(name)
        if not isinstance(other.get("limits"), dict):
            continue
        existing = source_limits(other)
        if existing["quota_group"] != new["quota_group"]:
            continue
        for key in ("per_minute", "per_hour", "per_day"):
            if existing[key] != new[key]:
                errors.append(f"limits.{key} is {new[key]} but source '{name}' of quota group "
                              f"'{new['quota_group']}' declares {existing[key]}; sources of the same group "
                              f"must declare the same limits")
    return errors


def estimate_initial_load(spec):
    """API cost of the initial load (full history) and the days the budget needs to complete it.
    One run per day can use at most the per-hour budget (a run lasts less than an hour) and the per-day budget."""
    limits = source_limits(spec)
    today = date.today()
    window = None if spec["load"]["strategy"] == "full" else history_window(spec["load"], today)
    total, chunks = 0.0, 0
    for ep in spec["endpoints"]:
        _, codes = resolve_locations(ep)
        for chunk_start, chunk_end in build_chunks(spec, ep, today, window):
            total += estimate_call_cost(limits, ep.get("params") or {}, chunk_start, chunk_end, len(codes or []))
            chunks += 1
    budget_per_run = min(limits["per_hour"], limits["per_day"]) * api_safety_margin
    return {"quota_group": limits["quota_group"], "chunks": chunks, "calls": round(total, 1),
            "budget_per_run": budget_per_run, "days": max(1, math.ceil(total / budget_per_run))}


def propose_source_spec(spec_yaml):
    """Agent tool: parse -> validate -> dry run -> human review -> save YAML -> initial load."""

    # 1. Parse YAML
    try:
        spec = load_source_spec(spec_yaml)
    except ValueError as e:
        print(f"🛡️ Spec rejected (YAML): {str(e)[:150]}")
        return json.dumps({
            "status": "rejected", "stage": "yaml", "errors": [str(e)],
            "instruction": "Fix the YAML syntax. Quote values that start with { or %, or that contain ': '.",
        })

    # 2. Validate rules + name not already taken + API limits
    errors = validate_source_spec(spec)
    if spec.get("source_name") in list_source_specs():
        errors.append(f"source '{spec['source_name']}' already exists; choose another source_name or ask the user")
    if "limits" not in spec:
        errors.append("limits is required for new sources: declare the provider limits (per_minute, per_hour, "
                      "per_day, cost, terms_url) or, if the provider publishes none, prudent values "
                      "(per_minute 60, per_hour 1000, per_day 5000)")
    if not errors:
        errors += quota_group_conflicts(spec)
    if errors:
        print(f"🛡️ Spec rejected by validation: {errors}")
        return json.dumps({
            "status": "rejected", "stage": "validation", "errors": errors,
            "instruction": "Change the spec according to the errors. Do not resend the same spec.",
        })

    # 3. Dry run: the API must answer AND return data (unless the provider cannot be called now)
    try:
        summary = run_source_spec(spec, window_override=_dry_run_window(spec), dry_run=True, caller="dry_run")
    except Exception as e:
        return json.dumps({"status": "error", "stage": "dry_run", "errors": [str(e)[:500]]})

    deferred = [r for r in summary["results"] if r["status"] == "deferred"]
    if deferred:
        reason = deferred[0]["error_message"]
        print(f"⏳ Dry run not possible now: {reason}")
        return json.dumps({
            "status": "blocked", "stage": "dry_run", "errors": [reason],
            "instruction": "The provider cannot be called now (quota group blocked or API budget exhausted). "
                           "Do NOT change or resend the spec and do not probe again. Stop and tell the user the "
                           "exact reason and that the onboarding can be retried later.",
        })

    problems = [f"{r['endpoint_name']} {r['chunk_start']}->{r['chunk_end']}: {r['error_message']}"
                for r in summary["results"] if r["status"] == "error"]
    problems += [f"{r['endpoint_name']}: the API answered but returned no data; check URL, params and dates"
                 for r in summary["results"] if r["status"] == "dry_run_ok" and _no_data(r["sample"])]
    if not summary["results"]:
        problems.append("dry run produced no chunks; check the load windows and end_offset")
    if problems:
        print(f"🧪 Dry run failed, the agent will fix it: {problems}")
        return json.dumps({
            "status": "error", "stage": "dry_run", "errors": problems,
            "instruction": "Probe the endpoint again if needed, fix the spec and propose it again.",
        })

    # 4. Human review: only for specs that already work
    estimate = estimate_initial_load(spec)
    print("\n===== THE AGENT PROPOSES A NEW SOURCE =====\n")
    print(spec_to_yaml(spec))
    print(f"💰 API budget · quota group {estimate['quota_group']} · initial load: {estimate['chunks']} call(s), "
          f"~{estimate['calls']:g} provider calls · budget per run {estimate['budget_per_run']:g} "
          f"({api_safety_margin:.0%} of the limits) -> {estimate['days']} day(s) to complete\n")
    shown = set()
    for r in summary["results"]:
        if r["endpoint_name"] not in shown:
            shown.add(r["endpoint_name"])
            print(f"🧪 Dry run sample · {r['endpoint_name']} ({r['chunk_start']}->{r['chunk_end']}):")
            print(r["sample"][:400], "\n")

    answer = input("Approve this source? (y = approve / n = reject / or type your feedback): ").strip()

    if answer.lower() in ("n", "no", ""):
        return json.dumps({"status": "rejected_by_human",
                           "message": "The user did not approve the new source. Tell the user."})

    if answer.lower() not in ("y", "yes"):
        print(f"📝 Feedback sent to the agent: {answer}")
        return json.dumps({
            "status": "changes_requested",
            "feedback": answer,
            "instruction": "The human reviewer requested changes. Apply the feedback exactly, "
                           "probe again if needed, and propose the revised spec.",
        })

    # 5. Save the spec (versioned in Git) and run the initial load (within the API budget)
    path = save_source_spec_file(spec)
    print(f"\n💾 Spec saved: {path}\n")
    load_result = run_source_spec(spec)

    print(f"\n✅ Source {spec['source_name']} onboarded · {load_result['chunks_ok']} ok / "
          f"{load_result['chunks_error']} errors / {load_result['chunks_deferred']} deferred")
    return json.dumps({
        "status": "created",
        "source_name": spec["source_name"],
        "spec_file": path,
        "chunks_ok": load_result["chunks_ok"],
        "chunks_error": load_result["chunks_error"],
        "chunks_deferred": load_result["chunks_deferred"],
        "initial_load_days": estimate["days"],
        "bronze_tables": list(load_result["bronze_tables"].keys()),
    })

In [0]:
import os
from datetime import date

example_sources = ["ree", "open_meteo_archive", "public_holidays_es"]   # specs shown to the agent as examples


# ------------------------------------------------------------
# 1. Context for the agent (read live from the repository)
# ------------------------------------------------------------
def list_existing_sources():
    sources = []
    for name in list_source_specs():
        spec = load_source_spec_file(name)
        sources.append({
            "source_name": name,
            "description": spec.get("description"),
            "load": spec.get("load"),
            "limits": spec.get("limits"),
            "quota_group": source_limits(spec)["quota_group"],
            "endpoints": [{"name": ep.get("name"), "url": ep.get("url"), "params": ep.get("params", {}),
                           "locations": ep.get("locations")}
                          for ep in spec.get("endpoints", [])],
        })
    return json.dumps(sources, ensure_ascii=False, default=str)


def _example_specs_text():
    texts = []
    for name in example_sources:
        if name in list_source_specs():
            with open(os.path.join(find_source_specs_dir(), f"{name}.yml"), encoding="utf-8") as f:
                texts.append(f"# --- example: {name}.yml ---\n{f.read()}")
    return "\n".join(texts)


def _reference_files_text():
    """Reference files the agent can use in a locations block (read live from reference_data/)."""
    lines = []
    folder = find_reference_data_dir()
    for file_name in sorted(os.listdir(folder)):
        if not file_name.endswith(".csv"):
            continue
        rows = read_reference_csv(file_name)
        if rows and {"latitude", "longitude"} <= set(rows[0]):
            lines.append(f"- {file_name}: {len(rows)} rows, columns {list(rows[0].keys())}")
    return "\n".join(lines) or "- (none)"


# ------------------------------------------------------------
# 2. Deterministic onboarding report (facts come from the saved spec, not from the LLM)
# ------------------------------------------------------------
def _describe_load_strategy(spec):
    load = spec["load"]
    if load["strategy"] == "full":
        return "full: complete download on every run (one snapshot per run date; previous snapshots are kept)"
    unit = load["unit"]
    offset = load.get("end_offset", 0)
    if offset == 0:
        until = f"the current {unit} (up to today)"
    elif offset > 0:
        until = f"the current {unit} + {offset} (future data, e.g. forecasts or calendars)"
    else:
        until = f"the current {unit} - {-offset} (the source publishes with a delay)"
    history = (f"history since {load['history_start']}" if "history_start" in load
               else f"history of the last {load['initial_window']} {unit}(s)")
    reconcile = (f"; weekly reconciliation re-loads {load['reconcile_window']} {unit}(s)"
                 if "reconcile_window" in load else "")
    return (f"incremental by {unit}: {history}; the daily run re-loads the last {load['refresh_window']} {unit}(s)"
            f"{reconcile}; up to {until}; calendar-aligned chunks of {load['chunk_size']} {unit}(s). "
            f"Missed runs and failed chunks are retried automatically. "
            f"Re-loaded periods are overwritten (latest version wins); older periods are kept")


def _describe_limits(spec):
    limits = source_limits(spec)
    text = (f"quota group `{limits['quota_group']}` · {limits['per_minute']}/min, {limits['per_hour']}/hour, "
            f"{limits['per_day']}/day (the framework uses at most {api_safety_margin:.0%})")
    if limits.get("terms_url"):
        text += f" · terms: {limits['terms_url']}"
    return text


def build_onboarding_report(outcome):
    spec = load_source_spec_file(outcome["source_name"])
    lines = [
        f"### ✅ Source onboarded: `{spec['source_name']}`",
        f"- **Description:** {spec['description']}",
        f"- **Spec file:** `{outcome['spec_file']}` (commit it in Git to complete the review)",
        f"- **Load strategy:** {_describe_load_strategy(spec)}",
        f"- **API limits:** {_describe_limits(spec)}",
        "- **Endpoints:**",
    ]
    for ep in spec["endpoints"]:
        line = f"  - `{ep['name']}`: {ep['url']} ({ep['response_format']})"
        if ep.get("locations"):
            n_locations = len(read_reference_csv(ep["locations"]["reference"]))
            line += f" · {n_locations} locations from `reference_data/{ep['locations']['reference']}`"
        lines.append(line)
    initial = f"{outcome['chunks_ok']} chunk(s) ok, {outcome['chunks_error']} error(s)"
    if outcome.get("chunks_deferred"):
        initial += (f", {outcome['chunks_deferred']} deferred by the API budget (loaded automatically by the next "
                    f"runs; about {outcome.get('initial_load_days', '?')} day(s) in total)")
    lines += [
        f"- **Initial load:** {initial}",
        "- **Bronze tables:** " + ", ".join(f"`{t}`" for t in outcome["bronze_tables"]),
        "- **Schedule:** picked up by the daily ingestion job once the spec is committed to Git",
    ]
    return "\n".join(lines)


# ------------------------------------------------------------
# 3. Agent instructions
# ------------------------------------------------------------
onboarding_prompt_template = """You are the Source Onboarding Agent of a governed data platform (Databricks, medallion architecture).
Your job: turn a user's request for a new data source into a SOURCE SPEC (YAML). You never write code.
A deterministic framework validates your spec, runs a dry run, asks a human for review and only then ingests the data into bronze.
Today is __TODAY__.
Always write in English, whatever language the user writes in.

PROCESS
1. Call list_existing_sources first. Do not onboard a source that already exists.
   Reuse what existing sources already do (variable names, locations block, timezone, load settings, quota_group
   and limits of the same provider) whenever the user asks for something similar or "the same as" an existing source.
2. Identify the public API that provides the requested data, from your own knowledge. You cannot search the web.
   If you do not know a suitable API, ask the user for the API documentation URL instead of guessing.
3. Verify with probe_url before proposing: use real parameters and a SMALL range (1-3 periods)
   close to today (__TODAY__). Never probe with dates from previous years unless the user asks for them.
   For APIs with locations, probe with 1-2 coordinates only; the full list is added by the framework.
   Read the structure: check the response format, the date format the API expects and that data is actually returned.
   If a probe fails, read the http_status and error, adjust the parameters and try again (at most 3 attempts).
   probe_url respects the API budget of the provider: if it is rejected because the quota group is blocked or the
   budget is exhausted, stop probing and tell the user the exact reason. Never try other hosts or tricks to get
   around a limit.
4. Write the spec and call propose_source_spec with the full YAML.
   - status rejected or error: read the errors, probe again if needed, fix the spec and propose again.
   - status changes_requested: the human reviewer wrote feedback. Apply it exactly and propose the revised spec.
   - status blocked: the provider cannot be called now. Do not change the spec; stop.
   Never resend the same spec. You have at most __MAX_PROPOSALS__ proposals.
5. When the tool returns status created, rejected_by_human or blocked, stop and write the final answer.

HOW SOURCES RUN (so you choose sensible load settings)
- A daily job re-loads the last refresh_window units of every source (late corrections).
- A weekly reconciliation re-loads reconcile_window units (corrections and deletions further back).
- A manual "initial" run reloads the full history. Missed runs and failed chunks are retried automatically.
- Every API call goes through a budget per quota group (at most __SAFETY_MARGIN__ of the declared limits).
  What does not fit is deferred to the next runs, most recent data first: a large initial load may take several days.
  After an HTTP 429 the provider is blocked until the next day for every caller.

SPEC FORMAT (YAML)
source_name: snake_case, 3-41 chars, unique (e.g. open_meteo_forecast)
source_type: http_api                # the only supported type
description: "quoted text: what the data is, provider, coverage"
provider: "organisation that provides the data"
license: "license or terms of use (e.g. CC BY 4.0, attribution required)"
auth:
  type: none                         # only public APIs without authentication
load:
  strategy: incremental              # incremental (data by period) or full (complete download on every run)
  # --- incremental only (do not use these keys with full) ---
  unit: day                          # day | month | year: the period unit of ALL the values below
  history_start: "2025-01-01"        # EITHER the fixed start of the history (preferred when the API keeps old data)
  initial_window: 90                 # OR a number of units back (for APIs that only keep recent data, e.g. forecasts).
                                     #   Use exactly one of history_start / initial_window.
  refresh_window: 7                  # units RE-loaded by the daily run, to capture late corrections.
                                     #   It does NOT delete anything: older periods are kept.
  reconcile_window: 60               # optional: units re-loaded by the weekly reconciliation (>= refresh_window)
  chunk_size: 7                      # units per API call. day 1-31, month 1-12, year 1-5. Keep responses small.
  end_offset: 0                      # where the load ends, in units from the current one:
                                     #   0 = up to today; negative = the source publishes with a delay;
                                     #   positive = future data (forecasts, calendars). day -10..2, month -6..1, year -2..1
  date_format: "%Y-%m-%d"            # how {start} and {end} are written for this API:
                                     #   "%Y-%m-%d", "%Y%m%d", "%d/%m/%Y", "%Y-%m", "%Y%m" or "%Y"
endpoints:                           # one or more
  - name: snake_case
    url: https://...                 # https only, public host, NO query string; may contain {start} (e.g. a year in the path)
    method: GET                      # GET only
    response_format: json            # json or csv
    csv_delimiter: ";"               # optional, csv only: one of , ; | and tab
    params:                          # query parameters: strings or numbers
      some_param: value
      start_param: "{start}"         # the first day of the chunk, written with date_format
      end_param: "{end}"             # the last day of the chunk, written with date_format
    locations:                       # optional: for APIs that accept lists of coordinates
      reference: locations_es.csv    # a file from the AVAILABLE REFERENCE FILES below
      key_column: province_code      # column that identifies each location
      latitude_param: latitude       # API parameter that receives the comma-separated latitudes
      longitude_param: longitude     # API parameter that receives the comma-separated longitudes
limits:                              # REQUIRED for new sources: protects the provider from overuse
  quota_group: snake_case            # sources that share the same provider quota use the same group
                                     #   (reuse the group of an existing source of the same provider)
  per_minute: 600                    # provider limits from its terms; if it publishes none: 60 / 1000 / 5000
  per_hour: 5000
  per_day: 10000
  min_seconds_between_calls: 1       # politeness gap between calls
  cost:                              # optional: how the provider counts one request
    per_location: true               #   each location of a locations block counts as one call
    days_per_unit: 14                #   every N days of data count as one more call
    variables_param: hourly          #   param with the comma-separated variables ...
    variables_per_unit: 10           #   ... every N variables count as one more call
  terms_url: https://...             # page with the provider's terms and limits
Rules:
- incremental endpoints must use {start} (in the url or in params); with chunk_size > 1 they must also use {end}.
- full endpoints must not use any placeholder. {start} and {end} are the only placeholders.
- Values that change over time (e.g. a year in the URL path) are expressed with {start} and the right unit/date_format,
  never hardcoded.
- Chunks are aligned to the calendar (7-day chunks are Monday-Sunday weeks, months start on day 1, years on January 1).
  Re-runs OVERWRITE the same chunks: only the latest version of each period is kept
  (e.g. an earlier forecast for a date is replaced by a newer one).
- locations: the framework fills latitude_param and longitude_param from the reference file and stores the
  location order in bronze. Do NOT put those parameters in params, and never write coordinate lists by hand
  when a reference file covers the requested places (e.g. "all Spanish provinces").
- limits: sources of the same provider must use the same quota_group AND the same limits. If you do not know the
  provider limits, use the prudent values and say so in the final answer: the human reviewer can correct them.
- YAML quoting: quote any value that starts with { or %, that contains ': ', dates like "2025-01-01",
  and lists of numbers like "40.4,37.3".

AVAILABLE REFERENCE FILES (reference_data/)
__REFERENCE_FILES__

EXAMPLES (approved specs already in production)
__EXAMPLE_SPECS__

LIMITS OF THE FRAMEWORK
If the source needs something the spec cannot express (authentication/API key, POST, pagination,
XML/Excel/ZIP), do not force a hack that loses data.
Either propose a reasonable workaround AND explain its limitation, or explain which framework extension is needed.

FINAL ANSWER (in English)
- If the source was CREATED: the framework already shows the user a factual summary (source, endpoints,
  load strategy, limits, tables, schedule). Do NOT repeat it. Write only observations and limitations,
  at most 5 short bullet points, checking:
  1. Does overwriting re-loaded periods lose previous versions the user may want to keep (e.g. earlier forecasts)?
     Remember: refresh_window and reconcile_window only decide what is re-loaded; older periods are never deleted.
  2. Is the naming consistent with existing sources (variable names, units, locations, timezone)?
  3. History or coverage limits of the API.
  4. Where the API limits come from (the provider terms you know, or prudent defaults to be confirmed).
  5. Anything else the data owner should decide.
  Only state facts that follow from the spec or from what you probed; if you are not sure, say so.
- If the source was NOT created: explain why, ALWAYS including the exact technical reason
  (tool, URL, http_status and error message) so the user can investigate."""


def build_onboarding_prompt():
    return (onboarding_prompt_template
            .replace("__TODAY__", f"{date.today()} ({date.today().strftime('%A')})")
            .replace("__MAX_PROPOSALS__", str(max_spec_proposals))
            .replace("__SAFETY_MARGIN__", f"{api_safety_margin:.0%}")
            .replace("__REFERENCE_FILES__", _reference_files_text())
            .replace("__EXAMPLE_SPECS__", _example_specs_text()))

In [0]:
import mlflow

max_agent_steps = 20
max_spec_proposals = 5
max_tool_result_chars = 6000
final_statuses = ("created", "rejected_by_human", "blocked")   # after these the agent can only answer

_llm_client = None


def get_llm_client():
    """Creates the LLM client only when the agent runs (tests of the framework do not need LLM packages)."""
    global _llm_client
    if _llm_client is None:
        from databricks.sdk import WorkspaceClient
        mlflow.openai.autolog()
        _llm_client = WorkspaceClient().serving_endpoints.get_open_ai_client()   # OpenAI-compatible client
    return _llm_client


def extract_text(content):
    """Text from a model response (works with reasoning models that return blocks)."""
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    parts = []
    for block in content:
        block_type = block.get("type") if isinstance(block, dict) else getattr(block, "type", None)
        text = block.get("text") if isinstance(block, dict) else getattr(block, "text", None)
        if block_type == "text" and text:
            parts.append(text)
    return "\n".join(parts)


# ------------------------------------------------------------
# Tools exposed to the LLM
# ------------------------------------------------------------
onboarding_tools = [
    {"type": "function", "function": {
        "name": "list_existing_sources",
        "description": "Lists the sources already onboarded (name, description, load settings, API limits and quota "
                       "group, endpoints with URL, params and locations). Call it first to avoid duplicates and to "
                       "reuse existing settings.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "probe_url",
        "description": (
            "Safe exploratory GET to a public https API. Returns HTTP status, detected format, "
            "the JSON structure with example values (or a CSV preview) and a text sample. Nothing is stored. "
            "Put query parameters in params, never in the url. Use small ranges close to today. "
            "It respects the provider's API budget: a blocked or exhausted provider is not called."
        ),
        "parameters": {"type": "object", "properties": {
            "url": {"type": "string", "description": "https URL without query string"},
            "params": {"type": "object", "description": "Query parameters as key/value pairs (strings or numbers)"},
        }, "required": ["url"]},
    }},
    {"type": "function", "function": {
        "name": "propose_source_spec",
        "description": (
            "Submits a complete source spec as YAML text. The framework validates it, runs a dry run "
            "and asks the human reviewer, who can approve, reject or request changes. "
            "If approved, it saves the spec and runs the initial load. Returns errors or feedback to act on."
        ),
        "parameters": {"type": "object", "properties": {
            "spec_yaml": {"type": "string", "description": "The full YAML spec"},
        }, "required": ["spec_yaml"]},
    }},
]


@mlflow.trace(span_type="TOOL")
def execute_onboarding_tool(tool_name, arguments, state):
    try:
        if tool_name == "list_existing_sources":
            result = list_existing_sources()
        elif tool_name == "probe_url":
            result = json.dumps(probe_url(arguments.get("url", ""), arguments.get("params")),
                                ensure_ascii=False, default=str)
        elif tool_name == "propose_source_spec":
            if state["proposals"] >= max_spec_proposals:
                state["finished"] = True
                result = json.dumps({"status": "rejected",
                                     "errors": [f"maximum of {max_spec_proposals} proposals reached"],
                                     "instruction": "Stop and explain to the user what failed."})
            else:
                state["proposals"] += 1
                result = propose_source_spec(arguments.get("spec_yaml", ""))
                outcome = json.loads(result)
                if outcome.get("status") == "created":
                    state["created"] = outcome
                if outcome.get("status") in final_statuses:
                    state["finished"] = True
        else:
            result = json.dumps({"error": f"Unknown tool {tool_name}"})
    except Exception as e:
        result = json.dumps({"error": f"{type(e).__name__}: {str(e)[:500]}"})
    return result[:max_tool_result_chars]


def _describe_call(tool_name, arguments):
    if tool_name == "probe_url":
        return f"{arguments.get('url')} {arguments.get('params') or ''}"
    if tool_name == "propose_source_spec":
        return f"(YAML, {len(str(arguments.get('spec_yaml', '')).splitlines())} lines)"
    return ""


def _describe_result(result):
    """One-line summary of a tool result, so the human can follow what the agent sees."""
    try:
        data = json.loads(result)
    except ValueError:
        return "→ (long result)"
    if isinstance(data, list):
        return f"→ {len(data)} item(s)"
    parts = [str(data[k]) for k in ("status", "stage", "http_status", "detected_format") if data.get(k) is not None]
    problem = data.get("errors") or data.get("error") or data.get("feedback") or data.get("warning") or data.get("hint")
    if problem:
        parts.append(str(problem)[:200])
    return "→ " + " · ".join(parts)


# ------------------------------------------------------------
# The agent loop
# ------------------------------------------------------------
@mlflow.trace(span_type="AGENT")
def run_onboarding_agent(request, max_steps=max_agent_steps):
    llm_client = get_llm_client()
    reset_probe_budget()
    state = {"proposals": 0, "finished": False, "created": None}
    messages = [{"role": "system", "content": build_onboarding_prompt()},
                {"role": "user", "content": request}]
    previous_calls = set()

    for step in range(max_steps):
        # Once the human approved/rejected (or the provider is blocked), the agent can only answer
        choice = "none" if state["finished"] else "auto"
        response = llm_client.chat.completions.create(
            model=llm_endpoint, messages=messages, tools=onboarding_tools, tool_choice=choice,
        )
        message = response.choices[0].message

        if not message.tool_calls:
            answer = extract_text(message.content)
            if state["created"]:
                return build_onboarding_report(state["created"]) + "\n\n#### Agent observations\n\n" + answer
            return answer

        messages.append({
            "role": "assistant",
            "content": extract_text(message.content),
            "tool_calls": [{"id": tc.id, "type": "function",
                            "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                           for tc in message.tool_calls],
        })

        for tc in message.tool_calls:
            try:
                arguments = json.loads(tc.function.arguments or "{}")
            except ValueError:
                arguments = None

            if arguments is None:
                result = json.dumps({"error": "Arguments are not valid JSON. Call the tool again with valid JSON."})
            else:
                call_key = (tc.function.name, json.dumps(arguments, sort_keys=True))
                if call_key in previous_calls:
                    print(f"[step {step + 1}] ⚠️ repeated call: {tc.function.name}")
                    result = json.dumps({"warning": "You already made this exact call; its result is earlier "
                                                    "in the conversation. Change something or write the final answer."})
                else:
                    previous_calls.add(call_key)
                    print(f"[step {step + 1}] {tc.function.name} {_describe_call(tc.function.name, arguments)}")
                    result = execute_onboarding_tool(tc.function.name, arguments, state)
                    print(f"           {_describe_result(result)}")

            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    return "Reached the maximum number of steps without a final answer."